# M15 · Clustering & cohort/persona discovery

Curriculum · Domain 3 · Unsupervised learning

**Find structure without labels, then test whether the structure is stable enough to use.**

We will create creator-style feature vectors, run k-means, compare $k$ with silhouette, inspect GMM soft assignments, and use DBSCAN to mark noise. Run top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(15)

## Build a small persona dataset

Each row is a creator or content account. The columns are standardized signals like technical depth, creator reach, buyer intent, and entertainment style.

In [ ]:
centers = np.array([
    [2.2, 0.4, 1.8, 0.2],
    [-1.5, 1.8, -0.8, 1.4],
    [0.1, -1.8, 1.2, -1.2],
])

X, true_group = make_blobs(
    n_samples=450,
    centers=centers,
    cluster_std=[0.55, 0.70, 0.60],
    random_state=15,
)

X = StandardScaler().fit_transform(X)

print("rows:", X.shape[0])
print("features:", X.shape[1])

## k-means objective

k-means chooses centroids $\mu_1,\ldots,\mu_k$ to minimize

$$\sum_{j=1}^k \sum_{x_i \in C_j} \|x_i - \mu_j\|_2^2.$$

The `inertia_` attribute below is exactly that within-cluster SSE.

In [ ]:
model = KMeans(n_clusters=3, random_state=15, n_init=20)
labels = model.fit_predict(X)
score = silhouette_score(X, labels)

print("SSE:", round(model.inertia_, 2))
print("silhouette:", round(score, 3))

assert score > 0.45

## Choose $k$ with two imperfect signals

The elbow asks where SSE stops falling quickly. Silhouette asks whether points are nearer to their own cluster than to other clusters. Neither is truth; together they are a useful diagnostic.

In [ ]:
ks = [2, 3, 4, 5, 6]
inertias = []
silhouettes = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=15, n_init=20)
    yk = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, yk))

for k, inertia, sil in zip(ks, inertias, silhouettes):
    print(k, round(inertia, 1), round(sil, 3))

## Visualize the first two feature axes

A plot is not a proof, but it is a fast way to catch obviously broken clusters before deeper review.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
scatter = ax.scatter(X[:, 0], X[:, 1], c=labels, s=20, cmap="viridis")
ax.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1], c="red", s=120, marker="x")
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.set_title("k-means creator personas")
plt.show()

## GMM: soft assignment instead of hard membership

A creator can look 70% like a technical educator and 30% like a career coach. GMM responsibilities capture that ambiguity.

In [ ]:
gmm = GaussianMixture(n_components=3, random_state=15)
gmm.fit(X)

probs = gmm.predict_proba(X[:5])

print(np.round(probs, 3))

row_sums = probs.sum(axis=1)
assert np.allclose(row_sums, 1.0)

## DBSCAN: density and noise

Density methods are useful when some inventory should not be forced into a persona. DBSCAN is the scikit-learn version; HDBSCAN extends the idea to variable density.

In [ ]:
noise = rng.uniform(low=-5.0, high=5.0, size=(18, X.shape[1]))
X_with_noise = np.vstack([X, noise])

db = DBSCAN(eps=0.75, min_samples=8)
db_labels = db.fit_predict(X_with_noise)

noise_count = int(np.sum(db_labels == -1))
cluster_count = len(set(db_labels)) - int(-1 in db_labels)

print("clusters:", cluster_count)
print("noise points:", noise_count)

assert noise_count >= 10

## Practice

1. Change `n_clusters` to 2 and 4. Which has the better silhouette?
2. Increase `cluster_std` and re-run. Watch silhouette fall as personas overlap.
3. Change DBSCAN `eps`. How many points become noise?
4. Inspect the GMM probability rows. Which creators are ambiguous?

In [ ]:
# Your turn
